In [1]:
!pip install -q onnx onnxscript onnxruntime

In [2]:
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np, onnxruntime as ort

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# MNIST 的標準化參數，之後板子上的前處理要用同一組數字
norm = transforms.Normalize((0.1307,), (0.3081,))
# 訓練時加入隨機旋轉、平移、縮放，讓模型能適應網頁上手寫的歪斜數字
train_tf = transforms.Compose([
    transforms.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.8, 1.2)),
    transforms.ToTensor(), norm])
test_tf = transforms.Compose([transforms.ToTensor(), norm])

train_ds = datasets.MNIST("data", train=True, download=True, transform=train_tf)
test_ds  = datasets.MNIST("data", train=False, download=True, transform=test_tf)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2)
test_dl  = DataLoader(test_ds, batch_size=512)

class DigitCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)   # 28x28
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)  # 14x14
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.drop = nn.Dropout(0.3)
    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)   # -> 14x14
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)   # -> 7x7
        x = x.flatten(1)
        x = self.drop(F.relu(self.fc1(x)))
        return self.fc2(x)

model = DigitCNN().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(5):
    model.train()
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        loss = F.cross_entropy(model(x), y)
        loss.backward()
        opt.step()
    model.eval(); correct = 0
    with torch.no_grad():
        for x, y in test_dl:
            correct += (model(x.to(device)).argmax(1).cpu() == y).sum().item()
    print(f"epoch {epoch+1}: loss={loss.item():.4f}  test_acc={correct/len(test_ds):.4f}")

# 匯出成 ONNX
model.eval().cpu()
dummy = torch.zeros(1, 1, 28, 28)
torch.onnx.export(model, (dummy,), "digit_cnn.onnx",
                  input_names=["input"], output_names=["logits"])
torch.save(model.state_dict(), "digit_cnn.pt")

# 驗證：PyTorch 和 ONNX 對同一張圖的輸出要一致
x, y = test_ds[0]
with torch.no_grad():
    pt_out = model(x.unsqueeze(0)).numpy()
sess = ort.InferenceSession("digit_cnn.onnx")
onnx_out = sess.run(None, {"input": x.unsqueeze(0).numpy()})[0]
print("label:", y, "| pytorch:", pt_out.argmax(), "| onnx:", onnx_out.argmax(),
      "| max diff:", np.abs(pt_out - onnx_out).max())

device: cuda


100%|██████████| 9.91M/9.91M [00:00<00:00, 13.3MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 397kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.69MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 17.1MB/s]


epoch 1: loss=0.2989  test_acc=0.9727
epoch 2: loss=0.2094  test_acc=0.9835
epoch 3: loss=0.1074  test_acc=0.9851
epoch 4: loss=0.1578  test_acc=0.9878
epoch 5: loss=0.0385  test_acc=0.9906
[torch.onnx] Obtain model graph for `DigitCNN([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `DigitCNN([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
label: 7 | pytorch: 7 | onnx: 7 | max diff: 1.9073486e-06


In [3]:
!ls -la digit_cnn*
xs = np.stack([test_ds[i][0].numpy() for i in range(20)])
ys = np.array([test_ds[i][1] for i in range(20)])
np.savez("test_samples.npz", x=xs, y=ys)
print(xs.shape, ys)

-rw-r--r-- 1 root root  10695 Nov 17 00:55 digit_cnn.onnx
-rw-r--r-- 1 root root 827456 Nov 17 00:55 digit_cnn.onnx.data
-rw-r--r-- 1 root root 831273 Nov 17 00:55 digit_cnn.pt
(20, 1, 28, 28) [7 2 1 0 4 1 4 9 5 9 0 6 9 0 1 5 9 7 3 4]


In [4]:
import onnx, os
m = onnx.load("digit_cnn.onnx")              # 會自動把 .data 裡的權重一起讀進來
onnx.save_model(m, "digit_cnn.onnx", save_as_external_data=False)
os.remove("digit_cnn.onnx.data")

# 確認合併後還能正常推論
sess = ort.InferenceSession("digit_cnn.onnx")
print("onnx pred:", sess.run(None, {"input": xs[:5]})[0].argmax(1), "label:", ys[:5])
!ls -la digit_cnn*

InvalidArgument: [ONNXRuntimeError] : 2 : INVALID_ARGUMENT : Got invalid dimensions for input: input for the following indices
 index: 0 Got: 5 Expected: 1
 Please fix either the inputs/outputs or the model.

In [5]:
sess = ort.InferenceSession("digit_cnn.onnx")
preds = [int(sess.run(None, {"input": xs[i:i+1]})[0].argmax()) for i in range(5)]
print("onnx pred:", preds, "label:", ys[:5].tolist())
!ls -la digit_cnn*

onnx pred: [7, 2, 1, 0, 4] label: [7, 2, 1, 0, 4]
-rw-r--r-- 1 root root 837849 Nov 17 00:57 digit_cnn.onnx
-rw-r--r-- 1 root root 831273 Nov 17 00:55 digit_cnn.pt
